In [2]:
import os
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory
from datapizzai.tools import tool

# Carica variabili d'ambiente
load_dotenv()

@tool
def calcolatrice(espressione: str) -> str:
    """Esegue calcoli matematici sicuri.
    
    Args:
        espressione: Espressione matematica (es: "2 + 3 * 4")
    
    Returns:
        Risultato del calcolo o messaggio di errore
    """
    try:
        # Validazione sicurezza
        allowed_chars = set('0123456789+-*/(). ')
        if not all(c in allowed_chars for c in espressione):
            return "Errore: Caratteri non permessi"
        
        result = eval(espressione)
        return f"Risultato: {result}"
    except Exception as e:
        return f"Errore: {str(e)}"

In [3]:
def create_calculator_client():
    """Crea un client specializzato in calcoli matematici."""
    
    client = ClientFactory.create(
        provider="openai",                    # Provider AI
        api_key=os.getenv("OPENAI_API_KEY"),  # API key da .env
        model="gpt-4o",                       # Modello OpenAI
        system_prompt="""Sei un assistente matematico esperto.
        Usa sempre lo strumento 'calcolatrice' per eseguire operazioni matematiche.
        Fornisci spiegazioni chiare e dettagliate.""",
        temperature=1,
    )
    
    if not client:
        raise ValueError("❌ Impossibile creare client OpenAI")
    
    return client

In [4]:
# 1. Crea il client
client = create_calculator_client()

# 2. Definisci i tool disponibili
tools = [calcolatrice]

# 3. Esegui query con tool automatico
response = client.invoke(
    input="Sia $k$ la radice di $m + n$, dove $n$ e $m$ sono due numeri distinti naturali minori di $100$. Trova il massimo valore intero di $k$",
    tools=tools,
    tool_choice="auto"  # OpenAI sceglie automaticamente quando usare i tool
)

# 4. Gestisci i risultati
def execute_tool_calls(response, available_tools):
    """Esegue i function call usando i tool passati (non il contenuto testuale)."""
    tool_results = []
    tool_map = {t.name: t for t in available_tools}

    for call in getattr(response, "function_calls", []) or []:
        tool_name = getattr(call, "name", None)
        arguments = getattr(call, "arguments", {}) or {}

        print(f"🔧 Tool chiamato: {tool_name}")
        print(f"📋 Argomenti: {arguments}")

        if tool_name in tool_map:
            result = tool_map[tool_name](**arguments)
            tool_results.append(result)
            print(f"✅ Risultato: {result}")
        else:
            print(f"⚠️ Tool sconosciuto: {tool_name}")
    
    return tool_results

# 5. Esegui i tool e mostra risultati
tool_results = execute_tool_calls(response, tools)

# 6. Mostra risposta finale
if response.text.strip():
    print(f"🤖 Assistente: {response.text}")
elif tool_results:
    print(f"🤖 Assistente: {tool_results[0]}")

🔧 Tool chiamato: calcolatrice
📋 Argomenti: {'espressione': '14^2'}
✅ Risultato: Errore: Caratteri non permessi
🔧 Tool chiamato: calcolatrice
📋 Argomenti: {'espressione': '15^2'}
✅ Risultato: Errore: Caratteri non permessi
🤖 Assistente: Per trovare il valore massimo intero di \( k \), dove \( k = \sqrt{m + n} \), con \( m \) e \( n \) numeri distinti naturali minori di 100, dobbiamo massimizzare l'espressione sotto radice, ossia \( m + n \).

Sappiamo che \( m \) e \( n \) possono raggiungere valori fino a 99, quindi il massimo valore che \( m + n \) può assumere è:

\[
m + n = 99 + 98 = 197
\]

Ora, dobbiamo trovare il più grande numero intero \( k \) tale che:

\[
k^2 \leq 197
\]

Calcoliamo i quadrati dei numeri interi vicini a \( \sqrt{197} \):

- \( 14^2 = 196 \)
- \( 15^2 = 225 \)

Dunque, \( k = 14 \) è la più grande radice intera che soddisfa la condizione \( k^2 \leq 197 \).

Implementiamo la verifica finale per assicurare la correttezza del calcolo computando \( 14^2 \) e \( 1

In [11]:
# 1. Definisci strumenti aggiuntivi
from datapizzai.tools.google import google_search_tool

response = client.invoke("Chi ha vinto wimbledon 2025?", tools=[google_search_tool])
@tool
def cerca_informazioni(query: str, max_results: int = 3, lang: str = "it") -> str:
    """Esegue una ricerca web reale (Bing Web Search API o SerpAPI).
    
    Args:
        query: Termine di ricerca
        max_results: Numero massimo di risultati da restituire
        lang: Lingua preferita (es. "it", "en")
    
    Returns:
        Un elenco sintetico di risultati con titolo e URL
    """
    import os
    import requests

    # Scegli motore: SERPAPI o Bing (in base a chiave disponibile)
    serpapi_key = os.getenv("SERPAPI_API_KEY")
    bing_key = os.getenv("BING_SEARCH_API_KEY")

    try:
        results = []
        if serpapi_key:
            # SerpAPI Web Search
            params = {
                "engine": "google",
                "q": query,
                "hl": lang,
                "num": max_results,
                "api_key": serpapi_key,
            }
            r = requests.get("https://serpapi.com/search", params=params, timeout=20)
            r.raise_for_status()
            data = r.json()
            for item in (data.get("organic_results") or [])[:max_results]:
                title = item.get("title")
                link = item.get("link")
                if title and link:
                    results.append(f"- {title} — {link}")
        else:
            return (
                "⚠️ Nessuna chiave trovata per la ricerca web. "
                "Configura SERPAPI_API_KEY o BING_SEARCH_API_KEY nel file .env"
            )

        if not results:
            return f"Nessun risultato per '{query}'"
        return "Risultati:\n" + "\n".join(results)

    except Exception as e:
        return f"Errore nella ricerca: {e}"


# 2. Crea client multi-tool
def create_multi_tool_client():
    """Crea un client con accesso a tutti gli strumenti."""
    
    client = ClientFactory.create(
        provider="openai",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4o",
        system_prompt="""Sei un assistente AI versatile con accesso a strumenti specializzati:

        - calcolatrice: per operazioni matematiche
        - cerca_informazioni: per ricerche web simulate  

        Analizza ogni richiesta e scegli lo strumento più appropriato.
        Per task complessi, puoi usare più strumenti in sequenza.
        Spiega sempre cosa stai facendo e perché."""
    )
    
    return client

# 3. Configura tutti i tool
tools = [calcolatrice, cerca_informazioni, gestisci_file]

# 4. Esegui workflow complessi
client = create_multi_tool_client()

complex_query = """
Esegui questo workflow:
1. Calcola quanti anni sono passati dal 1990 al 2025
2. Cerca informazioni su machine learning
"""

response = client.invoke(
    input=complex_query,
    tools=tools,
    tool_choice="auto"
)

# Il modello OpenAI sceglierà automaticamente i tool necessari
tool_results = execute_tool_calls(response, tools)

🔧 Tool chiamato: calcolatrice
📋 Argomenti: {'espressione': '2025 - 1990'}
✅ Risultato: Risultato: 35
🔧 Tool chiamato: cerca_informazioni
📋 Argomenti: {'query': 'machine learning', 'max_results': 3, 'lang': 'it'}
✅ Risultato: Risultati:
- Machine learning — https://en.wikipedia.org/wiki/Machine_learning
- What Is Machine Learning (ML)? — https://www.ibm.com/think/topics/machine-learning
- Machine Learning Crash Course — https://developers.google.com/machine-learning/crash-course


In [13]:
from datapizzai.type import TextBlock, ROLE
from datapizzai.memory import Memory

def run_with_tools(client, prompt, tools, max_steps=5):
    tool_map = {t.name: t for t in tools}
    memory = Memory()
    memory.add_turn([TextBlock(content=prompt)], ROLE.USER)

    response = client.invoke(input="", tools=tools, tool_choice="auto", memory=memory)
    steps = 0

    while getattr(response, "function_calls", []) and steps < max_steps:
        calls = response.function_calls or []
        results_text = []

        for call in calls:
            name = getattr(call, "name", "")
            args = getattr(call, "arguments", {}) or {}
            print(f"🔧 Tool chiamato: {name}")
            print(f"📋 Argomenti: {args}")

            if name in tool_map:
                res = tool_map[name](**args)
                print(f"✅ Risultato: {res}")
                results_text.append(f"{name}: {res}")
            else:
                results_text.append(f"{name}: Tool non trovato")

        # Fallback generico: rimanda i risultati al modello come testo in memoria
        memory.add_turn([TextBlock(content="Risultati strumenti:\n" + "\n".join(results_text))], ROLE.ASSISTANT)

        response = client.invoke(input="", tools=tools, tool_choice="auto", memory=memory)
        steps += 1

    print(f"🤖 Assistente: {response.text}")
    return response

In [15]:
from datapizzai.memory import Memory
from datapizzai.type import TextBlock, ROLE

def create_conversational_client():
    """Crea un client conversazionale con memoria."""
    
    # 1. Inizializza la memoria
    memory = Memory()
    
    # 2. Crea client con system prompt per conversazioni
    client = ClientFactory.create(
        provider="openai",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4o",
        system_prompt="""Sei un assistente AI amichevole con memoria conversazionale.
        Ricorda i dettagli delle conversazioni precedenti e fai riferimento ad essi quando appropriato.
        Usa gli strumenti disponibili per aiutare l'utente con task specifici."""
    )
    
    return client, memory

# 3. Configura conversazione multi-turno
client, memory = create_conversational_client()
tools = [calcolatrice, cerca_informazioni, gestisci_file]

def chat_turn(user_input: str, memory: Memory, client, tools):
    """Gestisce un turno: aggiorna memoria, invoca il client, esegue function calls."""
    
    print(f"👤 Utente: {user_input}")
    
    # Aggiungi input utente alla memoria
    memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
    
    # Invoca il client con memoria e tool
    response = client.invoke(
        input="",
        memory=memory,
        tools=tools,
        tool_choice="auto"
    )
    
    # NON aggiungere mai response.content alla memoria se ci sono function_calls
    tool_calls = getattr(response, "function_calls", []) or []
    
    if tool_calls:
        # Esegui i tool (riusa la tua funzione)
        tool_results = execute_tool_calls(response, tools)
    
        # Re-invoca passando i risultati come testo (niente tool_calls in memoria)
        followup = client.invoke(
            input="Usa questi risultati degli strumenti per completare la risposta:\n" + "\n".join(map(str, tool_results)),
            memory=memory,
            tools=tools,
            tool_choice="auto"
        )
    
        # Aggiungi solo testo finale alla memoria
        memory.add_turn([TextBlock(content=followup.text)], ROLE.ASSISTANT)
        print(f"🤖 Assistente: {followup.text}")
    
    else:
        # Nessun tool: salva normalmente
        memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        print(f"🤖 Assistente: {response.text}")
    
    #return response

# 4. Esempio di conversazione multi-turno
conversation = [
    "Ciao! Sono Mirko, sto lavorando su un progetto AI",
    "Cerca informazioni sui framework Python per AI",
    "Calcola il costo se spendo 500€ al mese per 2 anni",
    "Crea un file di progetto chiamato ai_project.txt",
    "Ricordi il mio nome e cosa sto facendo?"
]

for user_input in conversation:
    chat_turn(user_input, memory, client, tools)
    print()  # Spazio tra turni

# 5. Statistiche conversazione
print(f"📊 Turni totali: {len(memory.memory)}")
print(f"💬 Blocchi totali: {len(list(memory.iter_blocks()))}")

👤 Utente: Ciao! Sono Mirko, sto lavorando su un progetto AI
🤖 Assistente: Ciao Mirko! È fantastico che tu stia lavorando su un progetto AI. Di cosa si tratta? C'è qualcosa con cui posso aiutarti?

👤 Utente: Cerca informazioni sui framework Python per AI
🔧 Tool chiamato: cerca_informazioni
📋 Argomenti: {'query': 'framework Python per AI', 'lang': 'it'}
✅ Risultato: Risultati:
- Top 8 Python Libraries for Generative AI — https://datasciencedojo.com/blog/python-libraries-for-generative-ai/
- Top 7 Python Frameworks for AI Agents — https://www.kdnuggets.com/top-7-python-frameworks-for-ai-agents
🤖 Assistente: 

👤 Utente: Calcola il costo se spendo 500€ al mese per 2 anni
🔧 Tool chiamato: calcolatrice
📋 Argomenti: {'espressione': '500 * 24'}
✅ Risultato: Risultato: 12000
🤖 Assistente: Se spendi 500€ al mese per 2 anni, il costo totale sarà di 12.000€. Se hai bisogno di ulteriori calcoli o informazioni, fammi sapere!

👤 Utente: Crea un file di progetto chiamato ai_project.txt
🔧 Tool chiamato: